In [6]:
from __future__ import print_function
%matplotlib inline
import numpy
import matplotlib.pyplot as plt

# **Mixed Equations**
Sekarang kita mengeksplorasi bagaimana kita dapat menggunakan metode-metode yang telah kita analisis dan kembangkan untuk menyelesaikan persamaan yang lebih kompleks, yang tidak dengan mudah masuk ke dalam salah satu klasifikasi persamaan diferensial parsial (PDE) yang telah kita pelajari.

Kita akan memfokuskan pembahasan di sini pada persamaan diferensial parsial (PDE) yang berbentuk.
$$
    u_t = \mathcal{A}_1(u) + \mathcal{A}_2(u) + \cdots + \mathcal{A}_N(u)
$$
di mana $\mathcal{A}_j(u)$ adalah fungsi dari $u$ dan turunannya (yang juga mungkin bersifat nonlinier).

Karena sebagian besar metode yang akan kita bahas dapat digeneralisasi dari kasus dengan hanya dua operator $\mathcal{A}_j$ kita akan memfokuskan perhatian kita pada PDE
$$
    u_t = \mathcal{A}(u) + \mathcal{B}(u).
$$
Sekarang, mari kita pertimbangkan beberapa contoh dari jenis persamaan ini.



### Contoh – Masalah Multidimensi
Kita telah melihat bagaimana kita dapat mendekati masalah multidimensi yang dikombinasikan dengan turunan terhadap waktu. Masalah-masalah ini juga dianggap **mixed**, dan banyak metode yang akan kita bahas dapat diterapkan pada masalah multidimensi, seperti persamaan panas
$$
    u_t = \kappa(u_{xx} + u_{yy})
$$
atau PDE hiperbolik multidimensi
$$
    u_t + f(u)_x + g(u)_y = 0.
$$

### Contoh – Persamaan Reaksi-Difusi
Kita dapat menambahkan sebuah suku lain pada persamaan panas yang sering merepresentasikan suku reaksi kimia (juga kadang disebut sebagai sumber atau suku sinkronisasi), sehingga persamaannya menjadi
$$
    u_t = \kappa u_{xx} + R(u).
$$
Kita mungkin ingin menangani suku $R(u)$ secara berbeda dibandingkan dengan suku difusi, misalnya jika memiliki skala waktu yang berbeda, mungkin tidak stiff, atau sulit diselesaikan jika dikombinasikan langsung dengan pendekatan kita terhadap persamaan panas.

### Contoh – Persamaan Adveksi-Difusi
Kita juga telah melihat kasus-kasus saat mempertimbangkan metode numerik untuk adveksi, di mana persamaan termodifikasi dapat merepresentasikan sistem adveksi-difusi dengan bentuk
$$
    u_t + a u_x = \kappa u_{xx}.
$$


Ternyata, jenis persamaan seperti ini jauh lebih umum daripada yang terlihat; bahkan, persamaan Navier-Stokes merupakan contoh dari sekumpulan persamaan adveksi-difusi yang terbatas (terbatas karena kondisi inkompresibilitas). Kita juga sering menemukan persamaan hiperbolik nonlinier dengan suku viskositas, seperti
$$
    u_t + f(u)_x = \kappa u_{xx}
$$
ketika mendekati permasalahan dinamika fluida secara umum.
Contoh lainnya adalah persamaan Burgers viskos
$$
    u_t + u u_x = \epsilon u_{xx}
$$

### Contoh – Persamaan Adveksi-Difusi-Reaksi
Mengapa tidak menggabungkan semuanya?
$$
    u_t + f(u)_x = \kappa u_{xx} + R(u)
$$
Jenis persamaan ini sering muncul dalam kasus aliran fluida reaktif. Misalnya, pemodelan pembakaran biasanya melibatkan 10–100 suku reaksi yang berbeda dengan skala waktu yang sangat beragam, sehingga menjadikannya masalah yang sangat sulit untuk diselesaikan.

### Contoh – Adveksi-Dispersi
Kita juga telah melihat persamaan termodifikasi dalam studi PDE hiperbolik yang mengandung suku advektif dan dispersif. Contoh lain dari jenis persamaan ini adalah persamaan Kortweg-de Vries (KdV)
$$
    u_t + u u_x = \nu u_{xxx}.
$$
Persamaan ini dapat diturunkan dari persamaan Euler yang memodelkan aliran fluida inkompresibel dan merepresentasikan sejumlah fenomena menarik, terutama gelombang soliton.

Persamaan serupa adalah persamaan Schrödinger nonlinier
$$
    i \Psi_t(x,t) = -\Psi_{xx}(x,t) + V(\Psi)
$$
di mana $V(\Psi)$ adalah potensial nonlinier.

### Contoh – Adveksi-Difusi-Dispersi-Hiperdifusi
Persamaan Kuramoto-Sivashinsky
$$
    u_t + \frac{1}{2} (u_x)^2 = -u_{xx} - u_{xxxx}
$$
merupakan contoh lain dari persamaan yang menarik untuk dipelajari. Awalnya, persamaan ini tampak ill-posed dan mungkin mengalami blow-up karena tanda pada suku difusi, namun kenyataannya hal itu tidak terjadi. Redaman yang tepat disediakan oleh suku transport (di sisi kanan) untuk menstabilkan persamaan ini.

## Metode Garis Penuh (Fully Coupled Method of Lines)
Pendekatan pertama yang akan kita pelajari adalah pendekatan yang sebelumnya telah kita perkenalkan. Kita mengasumsikan bahwa diskretisasi spasial diterapkan sepenuhnya pada semua suku spasial sehingga menghasilkan sistem persamaan berbentuk
$$
    U'(t) = F(U(t)).
$$
Pendekatan ini dapat berhasil dan memberikan fleksibilitas besar dalam hal orde akurasi dan stensil yang tersedia, tetapi dapat menghadapi masalah ketika beberapa suku di sisi kanan bersifat stiff sementara yang lain tidak. Contoh terbaik dari hal ini adalah persamaan adveksi-difusi, kecuali jika kekuatan relatif adveksi dibandingkan difusi (disebut bilangan Peclet) sangat mendukung salah satu suku dibanding yang lain.

## Metode Deret Taylor Terhubung Penuh (Fully Coupled Taylor Series Methods)
Kita juga dapat memanfaatkan deret Taylor untuk membangun metode bagi persamaan campuran. Pertimbangkan perluasan Taylor terhadap waktu
$$
    u(x, t + \Delta t) \approx u(x, t) + \Delta t u_t + \cdots,
$$
Jika kita menggantikan $u_t$ dengan sisi kanan persamaan, kita membentuk metode
$$
    U^{n+1}_j = U^n_j + \Delta t (A(U^n_j) + B(U^n_j))
$$
di mana $A$ dan $B$ adalah versi diskret yang sesuai dari $\mathcal{A}$ dan $\mathcal{B}$.

Kita dapat memperluas metode ini ke orde lebih tinggi dengan mempertahankan lebih banyak suku dalam perluasan Taylor. Pertimbangkan PDE hiperbolik dua dimensi
$$
    u_t + a u_x + b u_y = 0.
$$
Deret Taylor yang dipotong hingga orde kedua adalah
$$
    u(x, t + \Delta t) \approx u(x, t) + \Delta t u_t + \frac{\Delta t^2}{2} u_{tt} + \cdots,
$$
sehingga kita perlu menghitung suku $u_{tt}$.